# 모델 아키텍처 및 배치 사이즈를 입력하면 필요한 VRAM을 출력하는 계산기 만들기

> 본 노트북은 SYLLABUS.md에 기반하여 자동 생성된 뼈대(Skeleton) 파일입니다. 상세한 이론, 수식 및 코드는 추가로 구현되어야 합니다.

## 1. 개요 및 학습 목표
이 노트북에서는 해당 주제에 대한 핵심 개념을 다룹니다.

## 2. 핵심 이론 및 수학적 원리
- 수식 및 상세한 동작 원리를 여기에 기록합니다.

## 3. 실습 코드 구현
아래 셀을 통해 파이썬 및 관련 프레임워크 코드를 직접 작성하고 실행해 볼 수 있습니다.

In [ ]:
# 실습을 위한 기본 라이브러리 임포트
import tensorflow as tf
import numpy as np

print(f"TensorFlow Version: {tf.__version__}")


---
## Q1: 파라미터 메모리 계산기

주어진 모델 아키텍처 설정으로 모델 파라미터가 차지하는 메모리를 계산하는 함수를 작성하세요.

**요구사항:**
- `calc_param_memory(num_params, dtype_bytes=4)` 함수를 정의하세요.
- `num_params`: 전체 파라미터 수 (예: 7e9 for 7B)
- `dtype_bytes`: float32=4, float16=2, int8=1, int4=0.5
- 결과를 GB 단위로 반환하세요.
- 7B, 13B, 70B 모델에 대해 fp32, fp16, int8, int4 각각의 메모리를 표 형태로 출력하세요.

In [ ]:
# TODO: 코드를 직접 작성해 보세요.

<details>
<summary>💡 정답 보기</summary>

```python
def calc_param_memory(num_params, dtype_bytes=4):
    """파라미터 저장에 필요한 메모리 계산 (GB)"""
    return num_params * dtype_bytes / (1024**3)

# 주요 모델 크기
models = {'7B': 7e9, '13B': 13e9, '70B': 70e9}
dtypes = {'fp32': 4, 'fp16': 2, 'int8': 1, 'int4': 0.5}

print(f"{'모델':<8}", end='')
for d in dtypes:
    print(f"{d:>10}", end='')
print()
print('-' * 48)

for name, params in models.items():
    print(f"{name:<8}", end='')
    for dtype, bytes_ in dtypes.items():
        mem_gb = calc_param_memory(params, bytes_)
        print(f"{mem_gb:>9.1f}G", end='')
    print()
```
</details>

---
## Q2: 훈련 시 총 메모리 계산기

모델을 훈련할 때는 파라미터 외에도 그래디언트, 옵티마이저 상태, 활성화값이 메모리를 차지합니다. 전체 훈련 메모리를 추정하는 함수를 작성하세요.

**요구사항:**
- `calc_training_memory(num_params, optimizer='adam', batch_size=1, seq_len=2048, hidden=4096)` 함수를 작성하세요.
- **파라미터**: fp16 기준 2 bytes/param
- **그래디언트**: 파라미터와 동일한 크기
- **Adam 옵티마이저 상태**: fp32 기준 8 bytes/param (m, v 두 모멘트)
- **활성화값 추정**: `batch_size * seq_len * hidden * 4 bytes * 4` (레이어당 근사)
- 7B 모델, batch_size=4, seq_len=2048 기준 총 메모리를 GB로 출력하세요.

In [ ]:
# TODO: 코드를 직접 작성해 보세요.

<details>
<summary>💡 정답 보기</summary>

```python
def calc_training_memory(num_params, optimizer='adam', batch_size=1, seq_len=2048, hidden=4096, num_layers=32):
    # 파라미터 (fp16)
    param_mem = num_params * 2
    # 그래디언트 (fp16)
    grad_mem = num_params * 2
    # Adam 옵티마이저 상태 (fp32 m, v)
    optim_mem = num_params * 8 if optimizer == 'adam' else num_params * 4
    # 활성화값 (근사)
    act_mem = batch_size * seq_len * hidden * 4 * num_layers

    total = param_mem + grad_mem + optim_mem + act_mem
    total_gb = total / (1024**3)

    print(f"파라미터: {param_mem/(1024**3):.1f} GB")
    print(f"그래디언트: {grad_mem/(1024**3):.1f} GB")
    print(f"옵티마이저 상태 (Adam): {optim_mem/(1024**3):.1f} GB")
    print(f"활성화값 (추정): {act_mem/(1024**3):.1f} GB")
    print(f"\n총 예상 훈련 메모리: {total_gb:.1f} GB")
    return total_gb

print("=== 7B 모델 훈련 메모리 추정 ===")
calc_training_memory(7e9, optimizer='adam', batch_size=4, seq_len=2048, hidden=4096, num_layers=32)
```
</details>

---
## Q3: GPU 수 결정 및 병렬화 전략 추천기

주어진 총 메모리 요구량과 GPU당 VRAM 기준으로, 몇 개의 GPU가 필요한지 계산하고 병렬화 전략을 추천하는 함수를 작성하세요.

**요구사항:**
- `recommend_strategy(total_mem_gb, gpu_vram_gb=80)` 함수를 정의하세요.
- 필요 GPU 수를 계산하고 (올림), 이에 맞는 병렬화 전략을 추천하세요.
- 1개 → 단일 GPU, 2~8개 → DDP, 8~32개 → DDP + Tensor Parallel, 32개 이상 → 3D Parallel 제안하세요.

In [ ]:
# TODO: 코드를 직접 작성해 보세요.

<details>
<summary>💡 정답 보기</summary>

```python
import math

def recommend_strategy(total_mem_gb, gpu_vram_gb=80):
    num_gpus = math.ceil(total_mem_gb / (gpu_vram_gb * 0.9))  # 10% 오버헤드 고려

    if num_gpus <= 1:
        strategy = "단일 GPU 학습"
    elif num_gpus <= 8:
        strategy = "DDP (Data Parallel)"
    elif num_gpus <= 32:
        strategy = "DDP + Tensor Parallel (TP)"
    else:
        strategy = "3D Parallel (PP + TP + DP)"

    print(f"필요 VRAM: {total_mem_gb:.1f} GB")
    print(f"GPU당 VRAM: {gpu_vram_gb} GB")
    print(f"필요 GPU 수: {num_gpus}개")
    print(f"추천 전략: {strategy}")
    return num_gpus, strategy

# 7B 모델 (앞서 계산한 값 사용)
recommend_strategy(total_mem_gb=180, gpu_vram_gb=80)
```
</details>